# M9 L9: RAG End-to-End

**Compose the M5-M8 primitives into a full RAG pipeline.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

**RAG** (Retrieval-Augmented Generation) is the production pattern for "answer questions about a private document collection."

The recipe: when the user asks a question, **search** the documents for the most relevant paragraphs, **paste** them into the prompt, and ask the AI to answer using only those paragraphs. This is the difference between "AI knows what the internet knew in 2021" and "AI knows your company's 2024 policies."

## The 5 things to know

1. **RAG is composition** -- ingest (M5) + chunk (M6) + embed (M3) + retrieve (M7) + prompt (M8).
2. **The four seams that bite**: ingestion, chunking, retrieval, prompt.
3. **Context cap** -- typically 2K-4K chars; longer dilutes signal and increases cost.
4. **Retrieval is the most common failure** -- fix with a reranker (M7).
5. **Always measure on a golden set** -- retrieval recall@K, end-to-end answer quality, cost, latency.

In [ ]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm09-rag-e2e' / 'L9-rag-end-to-end.py'
_spec = importlib.util.spec_from_file_location('L9', _p)
_L9 = importlib.util.module_from_spec(_spec)
sys.modules['L9'] = _L9
_spec.loader.exec_module(_L9)

print(f'Setup OK. Lesson module: {_L9.LESSON_NUMBER}')

In [ ]:
# Run the 13 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L9, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

In [ ]:
# === Section 1: build the RAG pipeline ===
rag = _L9.RAGPipeline()
n_chunks = rag.ingest(_L9.SAMPLE_DOCS, chunk_size=200, overlap=20)
print(f'Ingested: {n_chunks} chunks from {len(_L9.SAMPLE_DOCS)} docs')
print(f'Retriever size: {len(rag.retriever.chunks)} chunks')

In [ ]:
# === Section 2: query loop with retrieval recall@3 ===
n_correct = 0
results = []
for q, expected in _L9.SAMPLE_QUERIES:
    ans = rag.query(q, k=3)
    retrieved_ids = [c.doc_id for c in ans.retrieved_chunks]
    correct = expected in retrieved_ids
    n_correct += 1 if correct else 0
    results.append((q, expected, retrieved_ids, ans.answer, correct))
print(f'Retrieval recall@3: {n_correct}/{len(_L9.SAMPLE_QUERIES)} = {n_correct/len(_L9.SAMPLE_QUERIES):.0%}\n')
print(f'{"query":<46}  {"expected":<10}  {"top-1":<10}  hit?')
for q, exp, ids, ans, ok in results:
    print(f'  {q!r:<44}  {exp:<10}  {ids[0] if ids else "-":<10}  {"OK" if ok else "MISS"}')

In [ ]:
# === Section 3: sample answer (Q: 'how much vacation do I get per year') ===
ans = rag.query('how much vacation do I get per year', k=3)
print(f'Query:      {ans.query!r}')
print(f'Answer:     {ans.answer}')
print(f'Retrieved:  {[c.chunk_id for c in ans.retrieved_chunks]}')
print(f'Latency:    {ans.latency_ms:.2f} ms')
print(f'In tokens:  {ans.input_tokens}')
print(f'Out tokens: {ans.output_tokens}')
print(f'Cost:       ${ans.cost_usd:.6f}')

In [ ]:
# === Section 4: cost summary at scale ===
monthly = rag.total_cost_usd / rag.call_count * 1_000_000 if rag.call_count else 0
print(f'total calls:         {rag.call_count}')
print(f'total input tokens:  {rag.total_input_tokens}')
print(f'total output tokens: {rag.total_output_tokens}')
print(f'total cost USD:      ${rag.total_cost_usd:.6f}')
print(f'projected @ 1M/mo:   ${monthly:,.2f}')

In [ ]:
# === Section 5: chunking ablation (re-ingest with different sizes) ===
print('Re-ingest with different chunk sizes; report recall@3:\n')
for cs in (100, 200, 400, 800):
    rag2 = _L9.RAGPipeline()
    rag2.ingest(_L9.SAMPLE_DOCS, chunk_size=cs, overlap=20)
    n_ok = 0
    for q, expected in _L9.SAMPLE_QUERIES:
        ans = rag2.query(q, k=3)
        ids = [c.doc_id for c in ans.retrieved_chunks]
        if expected in ids:
            n_ok += 1
    print(f'  chunk_size={cs:>4}  recall@3 = {n_ok}/{len(_L9.SAMPLE_QUERIES)} = {n_ok/len(_L9.SAMPLE_QUERIES):.0%}')

In [ ]:
# === Section 6: top-K ablation ===
print('Sweep top-K on the original RAG; report recall:\n')
for k in (1, 3, 5, 10):
    n_ok = 0
    for q, expected in _L9.SAMPLE_QUERIES:
        ans = rag.query(q, k=k)
        ids = [c.doc_id for c in ans.retrieved_chunks]
        if expected in ids:
            n_ok += 1
    print(f'  k={k:>2}  recall = {n_ok}/{len(_L9.SAMPLE_QUERIES)} = {n_ok/len(_L9.SAMPLE_QUERIES):.0%}')

## Architecture (Mermaid)

```mermaid
flowchart LR
    Q[user query] --> E1[embed query]
    E1 --> R[retrieve top-K<br/>M7 hybrid]
    R --> CTX[build context<br/>cap at 2K chars]
    CTX --> P[prompt template<br/>M8]
    P --> L[LLM call<br/>M8]
    L --> A[answer text]
    A --> PARSE[output parser<br/>JSON / regex]
    PARSE --> OUT[structured answer]
    R --> LOG[retrieval log]
    L --> LOG2[LLM cost log]
    LOG --> OBS[observability]
    LOG2 --> OBS
```

**Read as:** Every arrow is a seam. Every seam is a place where the pipeline can break.

## Reflect (10 min)

1. Which of the four seams (ingestion, chunking, retrieval, prompt) is the most common failure mode in production? Why?
2. Why cap the context at 2K chars? What breaks at 8K? At 32K?
3. Walk through the cost math: 1M queries/month, 2K context, gpt-4o-mini. What's the monthly bill?
4. When would you choose fine-tuning over RAG? Give a concrete example.
5. (Stretch) Add a `citation_check(answer, retrieved_chunks)` that returns True if the answer's claims are supported by the retrieved context.

---

**Read next:**
- `lessons/m10-evaluation/L10-evaluation-ragas.ipynb` -- RAGAS eval framework
- `lessons/m11-observability/L11-observability-cost.ipynb` -- metrics, traces, alerts
- `SYLLABUS.md` -- the 18 modules and 21-day plan